# set up

In [1]:
#set up
import re
import string
import spacy
import pandas as pd


In [2]:
from google.colab import drive
drive.mount('/content/drive')
data = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/02DataForAnalysis/')
result = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/03Results/')
brief = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/04DataForR/')

Mounted at /content/drive


# ad (social media group)

## read the data

In [3]:
#read the data
ad_social = pd.read_csv(result + 'AD_social_result.csv',engine='python')
ad_social

,idx,creation_time,id,is_branded_content,lang,link_attachment.caption,link_attachment.description,link_attachment.link,link_attachment.name,media_type,...,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause,average_similarity
0,0,2024-09-16T08:00:05-07:00,1.194189e+15,False,en,NaN,NaN,NaN,NaN,albums,...,363.0,197.0,0.542700,28.0,1.120000,26.0,1.040000,28.0,1.120000,0.375187
1,1,2024-09-14T13:00:08-07:00,8.765100e+14,False,en,NaN,NaN,NaN,NaN,albums,...,294.0,177.0,0.602041,25.0,1.785714,28.0,2.000000,24.0,1.714286,0.368396
2,2,2024-09-13T01:00:06-07:00,5.042727e+14,False,en,NaN,NaN,NaN,NaN,albums,...,50.0,42.0,0.840000,9.0,3.000000,4.0,1.333333,1.0,0.333333,0.536808
3,3,2024-09-12T01:00:01-07:00,1.286549e+15,False,en,NaN,NaN,NaN,NaN,photos,...,573.0,282.0,0.492147,52.0,1.677419,62.0,2.000000,39.0,1.258065,0.329114
4,4,2024-09-09T05:00:01-07:00,1.231950e+15,False,en,NaN,NaN,NaN,NaN,photos,...,309.0,173.0,0.559871,36.0,2.400000,33.0,2.200000,16.0,1.066667,0.371303
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2675,2675,2010-04-15T07:17:43-07:00,5.187244e+14,False,NaN,alzheimers.org.uk,Alzheimer's Society called on our political pa...,http://www.alzheimers.org.uk/site/scripts/docu...,What are the three main political parties’ ele...,links,...,24.0,21.0,0.875000,2.0,1.000000,4.0,2.000000,2.0,1.000000,0.113050
2676,2676,2010-04-01T07:54:09-07:00,3.725374e+15,False,NaN,alzheimers.org.uk,NaN,http://www.alzheimers.org.uk/site/scripts/docu...,www.alzheimers.org.uk,links,...,32.0,29.0,0.906250,1.0,0.333333,2.0,0.666667,3.0,1.000000,0.309829
2677,2677,2010-03-31T08:29:41-07:00,8.861507e+14,False,NaN,forum.alzheimers.org.uk,"Many Happy Returns, TP Support for people with...",http://forum.alzheimers.org.uk/showthread.php?...,"Many Happy Returns, TP - Talking Point",links,...,9.0,9.0,1.000000,0.0,0.000000,0.0,0.000000,1.0,1.000000,NaN
2678,2678,2009-09-21T04:18:01-07:00,1.517941e+15,False,NaN,NaN,NaN,NaN,NaN,photos,...,26.0,24.0,0.923077,1.0,0.500000,2.0,1.000000,2.0,1.000000,0.522011


In [4]:
len(ad_social["id"])

2680

In [5]:
len(set(ad_social["id"])) #each id is a unique number

2680

## tokenize the text, and Explode the 'word_list' column

In [6]:
# Tokenize the text (split each sentence into words)
ad_social['word_list'] = ad_social['clean_text'].astype(str).apply(lambda x: x.split())
ad_social.head(2)

,idx,creation_time,id,is_branded_content,lang,link_attachment.caption,link_attachment.description,link_attachment.link,link_attachment.name,media_type,...,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause,average_similarity,word_list
0,0,2024-09-16T08:00:05-07:00,1.194189e+15,False,en,NaN,NaN,NaN,NaN,albums,...,197.0,0.542700,28.0,1.120000,26.0,1.04,28.0,1.120000,0.375187,"[shocking, new, data, has, shown, that, people..."
1,1,2024-09-14T13:00:08-07:00,8.765100e+14,False,en,NaN,NaN,NaN,NaN,albums,...,177.0,0.602041,25.0,1.785714,28.0,2.00,24.0,1.714286,0.368396,"[‘i, never, imagined, dementia, impacting, our..."


In [7]:
df=ad_social[['idx','id','clean_text','word_list']]
df

,idx,id,clean_text,word_list
0,0,1.194189e+15,shocking new data has shown that people living...,"[shocking, new, data, has, shown, that, people..."
1,1,8.765100e+14,‘i never imagined dementia impacting our lives...,"[‘i, never, imagined, dementia, impacting, our..."
2,2,5.042727e+14,we know there is a long way to go when it come...,"[we, know, there, is, a, long, way, to, go, wh..."
3,3,1.286549e+15,‘it felt like we had just been given a label a...,"[‘it, felt, like, we, had, just, been, given, ..."
4,4,1.231950e+15,"‘we had all heard about dementia, but we never...","[‘we, had, all, heard, about, dementia,, but, ..."
...,...,...,...,...
2675,2675,5.187244e+14,want to know what the major parties are saying...,"[want, to, know, what, the, major, parties, ar..."
2676,2676,3.725374e+15,don't miss out on one of the best races in the...,"[don't, miss, out, on, one, of, the, best, rac..."
2677,2677,8.861507e+14,'talking point' our online forum is 7 years ol...,"['talking, point', our, online, forum, is, 7, ..."
2678,2678,1.517941e+15,thanks to everyone that took part in this week...,"[thanks, to, everyone, that, took, part, in, t..."


In [8]:
# Explode the 'word_list' column, creating a separate row for each word, and renames the exploded column to 'word'
df_exploded = df.explode('word_list').rename(columns={'word_list': 'word'})
df_exploded.head()

,idx,id,clean_text,word
0,0,1.194189e+15,shocking new data has shown that people living...,shocking
0,0,1.194189e+15,shocking new data has shown that people living...,new
0,0,1.194189e+15,shocking new data has shown that people living...,data
0,0,1.194189e+15,shocking new data has shown that people living...,has
0,0,1.194189e+15,shocking new data has shown that people living...,shown


**lowcasing, remove quotation marks (single quotes, double quotes, and special quotes) from the start and end of the strings**

In [9]:
def clean_word_column(df):

    # Remove punctuation: Uses regex to remove standard punctuation (from string.punctuation) around each word.
    df['word'] = df['word'].str.replace(f"[{string.punctuation}]", "", regex=True)

    # Lowercasing: Converts each word to lowercase (we did this when preprocessing "text" column)
    # df['word_lower'] = df['word'].str.lower()

    # Quotation mark removal: Uses regex to remove single quotes, double quotes, and special quotes from the start and end of each word.
    df['word'] = df['word'].str.replace(r"^[‘’“”\"']|[‘’“”\"']$", '', regex=True)

    # Row filtering: Remove rows where 'word_lower' is NaN or contains non-string values
    df = df[df['word'].apply(lambda x: isinstance(x, str))]

    # Return the result
    return df


In [10]:
df_lex = clean_word_column(df_exploded)
df_lex[['idx','id','word']]

,idx,id,word
0,0,1.194189e+15,shocking
0,0,1.194189e+15,new
0,0,1.194189e+15,data
0,0,1.194189e+15,has
0,0,1.194189e+15,shown
...,...,...,...
2679,2679,8.267263e+14,having
2679,2679,8.267263e+14,a
2679,2679,8.267263e+14,grandparent
2679,2679,8.267263e+14,with


## use Spacy: tokenize words, and POS tagging

In [11]:
# Load the spaCy model for English,which includes tokenization, lemmatization, and POS tagging capabilities.
nlp = spacy.load("en_core_web_sm")

In [12]:
def process_word_spacy(df, text_column='word'):
    # Ensure column is in string format and not NaN
    df = df[df[text_column].notna()]
    df[text_column] = df[text_column].astype(str)

    # Create spaCy documents using nlp.pipe for efficient processing
    docs = list(nlp.pipe(df[text_column]))

    # Initialize lists to store processing results
    token_texts = []
    lemmas = []
    pos_tags = []
    tags = []
    token_counts = []

    # Process each document and extract relevant information
    for doc in docs:
        # Token text (original form), lemmas (base form), POS (simple and detailed)
        token_texts.append([token.text for token in doc])
        lemmas.append([token.lemma_ for token in doc])
        pos_tags.append([token.pos_ for token in doc])  # Simple POS tags
        tags.append([token.tag_ for token in doc])  # Detailed POS tags
        token_counts.append(len(doc))  # Number of tokens in each doc

    # Add the results as new columns in the DataFrame
    df = df.assign(
        sp_tokenized=token_texts,
        sp_n_token=token_counts,
        sp_lemma=lemmas,
        sp_pos=pos_tags,
        sp_tag=tags
    )

    return df

In [13]:
df_lex = process_word_spacy(df_lex)
df_lex.head()

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag
0,0,1.194189e+15,shocking new data has shown that people living...,shocking,[shocking],1,[shocking],[ADJ],[JJ]
0,0,1.194189e+15,shocking new data has shown that people living...,new,[new],1,[new],[ADJ],[JJ]
0,0,1.194189e+15,shocking new data has shown that people living...,data,[data],1,[datum],[NOUN],[NNS]
0,0,1.194189e+15,shocking new data has shown that people living...,has,[has],1,[have],[VERB],[VBZ]
0,0,1.194189e+15,shocking new data has shown that people living...,shown,[shown],1,[show],[VERB],[VBN]


In [14]:
# save the results to a new cvs. file
df_lex.to_csv(result + 'AD_social_lexicon.csv', index=False)

# Inform the user that the file was successfully saved
print(f"The file was successfully saved at: {result + 'AD_social_lexicon.csv'}")

The file was successfully saved at: /content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/03Results/AD_social_lexicon.csv


## Count the number of POS tags (group by id)

In [15]:
def count_pos_tags_by_speaker(df, pos_column='sp_pos', group_column='id'):

    # Ensure the specified columns exist in the DataFrame
    if pos_column not in df.columns or group_column not in df.columns:
        raise ValueError(f"Columns '{pos_column}' and '{group_column}' must exist in the DataFrame.")

    # Explode the pos_column to have one row per POS tag
    df_exploded = df.explode(pos_column)

    # Group by the specified group column and count each POS tag
    pos_counts = (
        df_exploded.groupby(group_column)[pos_column]
        .value_counts()
        .unstack(fill_value=0)  # Fill NaNs with 0s for missing POS tags
    )

    # Add a new column for the total POS count by summing across all POS columns
    pos_counts['total_pos'] = pos_counts.sum(axis=1)

    # Return the resulting DataFrame
    return pos_counts




```
    Count the number of each POS tag per speaker and calculate the total POS tags.
    
    Parameters:
    - df: DataFrame containing POS tags in a list format.
    - pos_column: Column containing lists of POS tags for each row (default: 'sp_pos').
    - group_column: Column to group by, such as speaker_id (default: 'speaker_id').

    Returns:
    - DataFrame with counts of each POS tag per speaker and a total POS count.
```



In [16]:
df_pos = count_pos_tags_by_speaker(df_lex)
df_pos.head()

sp_pos,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,SYM,VERB,X,total_pos
id,,,,,,,,,,,,,,,,,,
2.664801e+14,10,29,28,17,22,1,4,68,7,16,90,12,2,5,0,83,0,394
2.773841e+14,4,7,2,5,2,0,3,19,3,3,13,6,0,1,0,16,0,84
2.901395e+14,7,11,6,6,3,0,0,11,0,2,19,3,0,2,0,14,0,84
2.901494e+14,11,25,13,10,8,0,1,34,1,6,44,15,4,6,0,33,0,211
2.901739e+14,9,15,2,4,2,0,1,22,6,6,24,6,0,0,2,18,0,117


POS: The simple UPOS part-of-speech tag

Universal POS tag: https://universaldependencies.org/u/pos/

- ADJ: adjective.
- ADP: adposition.
- ADV: adverb.
- AUX: auxiliary.
- CCONJ: coordinating conjunction.
- DET: determiner.
- INTJ: interjection.
- NOUN: noun.
- NUM: numeral.
- PART: particle.
- PRON: pronoun.
- PROPN: proper noun.
- PUNCT: punctuation.
- SCONJ: subordinating conjunction.
- SYM: symbol.
- VERB: verb.
- X: other.

## Proportion of Part-of-speech

In [17]:
def pos_analysis(df):

    # Step 1: Create a new DataFrame with selected columns
    pos_df = df[[ 'ADJ', 'VERB', 'PRON', 'NOUN', 'total_pos']].copy()

    # Step 2: Calculate normalized (percentage) columns for each POS type
    pos_df['ADJ%'] = (pos_df['ADJ'] / pos_df['total_pos']) * 100
    pos_df['VERB%'] = (pos_df['VERB'] / pos_df['total_pos']) * 100
    pos_df['PRON%'] = (pos_df['PRON'] / pos_df['total_pos']) * 100
    pos_df['NOUN%'] = (pos_df['NOUN'] / pos_df['total_pos']) * 100

    # Step 3: Calculate the specified POS ratios
    pos_df['Noun_to_Verb'] = pos_df['NOUN'] / (pos_df['VERB'] )
    pos_df['Pron_to_Noun'] = pos_df['PRON'] / (pos_df['NOUN'] )
    pos_df['Noun_to_Adj'] = pos_df['NOUN'] / (pos_df['ADJ'])
    pos_df['Verb_to_Adj'] = pos_df['VERB'] / (pos_df['ADJ'])

    return pos_df

In [18]:
pos_result = pos_analysis(df_pos)
pos_result.head()

sp_pos,ADJ,VERB,PRON,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
id,,,,,,,,,,,,,
2.664801e+14,10,83,90,68,394,2.538071,21.065990,22.842640,17.258883,0.819277,1.323529,6.800000,8.3
2.773841e+14,4,16,13,19,84,4.761905,19.047619,15.476190,22.619048,1.187500,0.684211,4.750000,4.0
2.901395e+14,7,14,19,11,84,8.333333,16.666667,22.619048,13.095238,0.785714,1.727273,1.571429,2.0
2.901494e+14,11,33,44,34,211,5.213270,15.639810,20.853081,16.113744,1.030303,1.294118,3.090909,3.0
2.901739e+14,9,18,24,22,117,7.692308,15.384615,20.512821,18.803419,1.222222,1.090909,2.444444,2.0


In [19]:
pos_result.columns

Index(['ADJ', 'VERB', 'PRON', 'NOUN', 'total_pos', 'ADJ%', 'VERB%', 'PRON%',
       'NOUN%', 'Noun_to_Verb', 'Pron_to_Noun', 'Noun_to_Adj', 'Verb_to_Adj'],
      dtype='object', name='sp_pos')

In [24]:
# save the result to the orginal csv. file
# pd.merge(): Merges the two DataFrames on the common column 'id'
# how='left': Ensures that all rows from ad_social are kept, even if there’s no matching row in pos_result.

merged_df = pd.merge(ad_social, pos_result[['ADJ', 'VERB', 'PRON', 'NOUN', 'total_pos',
                                            'ADJ%', 'VERB%', 'PRON%','NOUN%',
                                            'Noun_to_Verb', 'Pron_to_Noun', 'Noun_to_Adj', 'Verb_to_Adj']],
                     on='id',
                     how='left')
merged_df

,idx,creation_time,id,is_branded_content,lang,link_attachment.caption,link_attachment.description,link_attachment.link,link_attachment.name,media_type,...,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
0,0,2024-09-16T08:00:05-07:00,1.194189e+15,False,en,NaN,NaN,NaN,NaN,albums,...,74.0,377.0,6.366048,20.159151,17.506631,19.628647,0.973684,0.891892,3.083333,3.166667
1,1,2024-09-14T13:00:08-07:00,8.765100e+14,False,en,NaN,NaN,NaN,NaN,albums,...,60.0,308.0,4.220779,17.532468,18.506494,19.480519,1.111111,0.950000,4.615385,4.153846
2,2,2024-09-13T01:00:06-07:00,5.042727e+14,False,en,NaN,NaN,NaN,NaN,albums,...,9.0,51.0,1.960784,21.568627,19.607843,17.647059,0.818182,1.111111,9.000000,11.000000
3,3,2024-09-12T01:00:01-07:00,1.286549e+15,False,en,NaN,NaN,NaN,NaN,photos,...,103.0,590.0,4.745763,18.305085,21.186441,17.457627,0.953704,1.213592,3.678571,3.857143
4,4,2024-09-09T05:00:01-07:00,1.231950e+15,False,en,NaN,NaN,NaN,NaN,photos,...,35.0,318.0,6.289308,17.295597,22.641509,11.006289,0.636364,2.057143,1.750000,2.750000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2675,2675,2010-04-15T07:17:43-07:00,5.187244e+14,False,NaN,alzheimers.org.uk,Alzheimer's Society called on our political pa...,http://www.alzheimers.org.uk/site/scripts/docu...,What are the three main political parties’ ele...,links,...,3.0,24.0,4.166667,25.000000,16.666667,12.500000,0.500000,1.333333,3.000000,6.000000
2676,2676,2010-04-01T07:54:09-07:00,3.725374e+15,False,NaN,alzheimers.org.uk,NaN,http://www.alzheimers.org.uk/site/scripts/docu...,www.alzheimers.org.uk,links,...,4.0,34.0,5.882353,29.411765,17.647059,11.764706,0.400000,1.500000,2.000000,5.000000
2677,2677,2010-03-31T08:29:41-07:00,8.861507e+14,False,NaN,forum.alzheimers.org.uk,"Many Happy Returns, TP Support for people with...",http://forum.alzheimers.org.uk/showthread.php?...,"Many Happy Returns, TP - Talking Point",links,...,4.0,10.0,10.000000,20.000000,10.000000,40.000000,2.000000,0.250000,4.000000,2.000000
2678,2678,2009-09-21T04:18:01-07:00,1.517941e+15,False,NaN,NaN,NaN,NaN,NaN,photos,...,8.0,26.0,3.846154,15.384615,26.923077,30.769231,2.000000,0.875000,8.000000,4.000000


In [21]:
# save the results to a new cvs. file
merged_df.to_csv(brief + 'AD_social_brief_result.csv', index=False)

let us also save the results to the "brief results"

# Health (social media group)

In [25]:
#read the data
health_social = pd.read_csv(result + 'Health_social_result.csv',engine='python')


In [26]:
health_social.head(2)

,idx,statistics.like_count,link_attachment.caption,statistics.love_count,post_owner.username,post_owner.name,modified_time,statistics.haha_count,surface.id,lang,...,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause,average_similarity
0,0,67.0,aarp.org,0.0,AARP,AARP,2024-10-02T10:48:06-07:00,0.0,6.971043e+14,en,...,49.0,41.0,0.836735,5.0,2.50,6.0,3.0,6.0,3.0,0.608770
1,1,96.0,aarp.org,3.0,AARP,AARP,2024-10-02T10:53:20-07:00,2.0,6.971043e+14,en,...,98.0,72.0,0.734694,3.0,0.75,10.0,2.5,12.0,3.0,0.412788


In [27]:
len(health_social["id"])

3073

In [28]:
len(set(health_social["id"])) #each id is a unique number

3073

**tokenize** the text, and Explode the 'word_list' column

In [29]:
# Tokenize the text (split each sentence into words)
health_social['word_list'] = health_social['clean_text'].astype(str).apply(lambda x: x.split())
health_social.head(2)

,idx,statistics.like_count,link_attachment.caption,statistics.love_count,post_owner.username,post_owner.name,modified_time,statistics.haha_count,surface.id,lang,...,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause,average_similarity,word_list
0,0,67.0,aarp.org,0.0,AARP,AARP,2024-10-02T10:48:06-07:00,0.0,6.971043e+14,en,...,41.0,0.836735,5.0,2.50,6.0,3.0,6.0,3.0,0.608770,"[aarp, volunteers, gathered, with, user, exper..."
1,1,96.0,aarp.org,3.0,AARP,AARP,2024-10-02T10:53:20-07:00,2.0,6.971043e+14,en,...,72.0,0.734694,3.0,0.75,10.0,2.5,12.0,3.0,0.412788,"[taraji, p., henson, stars, in, the, newly, la..."


In [30]:
df=health_social[['idx','id','clean_text','word_list']]
df

,idx,id,clean_text,word_list
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,"[aarp, volunteers, gathered, with, user, exper..."
1,1,1.527138e+15,taraji p. henson stars in the newly launched p...,"[taraji, p., henson, stars, in, the, newly, la..."
2,2,8.547888e+14,howie mandel found his purpose doing standup c...,"[howie, mandel, found, his, purpose, doing, st..."
3,3,1.220167e+15,who is the highest-grossing actor? samuel l. j...,"[who, is, the, highest-grossing, actor?, samue..."
4,4,8.840548e+14,legendary drummer and singer sheila e. has man...,"[legendary, drummer, and, singer, sheila, e., ..."
...,...,...,...,...
3068,3068,5.726419e+14,with more than 80% of older adults living with...,"[with, more, than, 80%, of, older, adults, liv..."
3069,3069,9.777879e+14,"neither the systems serving older adults, nor ...","[neither, the, systems, serving, older, adults..."
3070,3070,9.463259e+14,"in the fall of 2023, more than 40 indigenous e...","[in, the, fall, of, 2023,, more, than, 40, ind..."
3071,3071,1.773792e+15,webinar tomorrow | learn more about later-life...,"[webinar, tomorrow, |, learn, more, about, lat..."


In [31]:
# Explode the 'word_list' column, creating a separate row for each word, and renames the exploded column to 'word'
df_exploded = df.explode('word_list').rename(columns={'word_list': 'word'})
df_exploded.head()

,idx,id,clean_text,word
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,aarp
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,volunteers
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,gathered
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,with
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,user


**clean the "word" column**

lowcasing, remove quotation marks (single quotes, double quotes, and special quotes) from the start and end of the strings

In [32]:
df_lex = clean_word_column(df_exploded)
df_lex[['idx','id','word']]

,idx,id,word
0,0,1.087023e+15,aarp
0,0,1.087023e+15,volunteers
0,0,1.087023e+15,gathered
0,0,1.087023e+15,with
0,0,1.087023e+15,user
...,...,...,...
3072,3072,8.929571e+15,and
3072,3072,8.929571e+15,how
3072,3072,8.929571e+15,to
3072,3072,8.929571e+15,get


**use Spacy: tokenize words, and POS tagging**

In [33]:
df_lex = process_word_spacy(df_lex)
df_lex.head()

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,aarp,[aarp],1,[aarp],[NOUN],[NN]
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,volunteers,[volunteers],1,[volunteer],[NOUN],[NNS]
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,gathered,[gathered],1,[gather],[VERB],[VBD]
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,with,[with],1,[with],[ADP],[IN]
0,0,1.087023e+15,aarp volunteers gathered with user experience ...,user,[user],1,[user],[NOUN],[NN]


In [35]:
# save the results
df_lex.to_csv(result + 'Health_social_lexicon.csv', index=False)

In [36]:
# Count the number of POS tags (group by id)
df_pos = count_pos_tags_by_speaker(df_lex)
df_pos.head()

sp_pos,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,VERB,X,total_pos
id,,,,,,,,,,,,,,,,,
1.892159e+14,4,4,2,2,2,0,0,17,0,2,7,3,0,1,7,1,52
2.290554e+14,1,2,0,1,0,0,0,7,0,0,2,1,1,0,3,0,18
2.656471e+14,2,3,3,1,3,0,0,13,3,1,5,2,0,0,7,0,43
2.813105e+14,5,5,2,4,1,0,0,15,1,2,10,2,0,0,8,0,55
2.996404e+14,1,9,3,3,3,0,1,22,0,1,11,6,0,0,7,0,67


In [37]:
# count proportion of Part-of-speech
pos_result = pos_analysis(df_pos)
pos_result.head()

sp_pos,ADJ,VERB,PRON,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
id,,,,,,,,,,,,,
1.892159e+14,4,7,7,17,52,7.692308,13.461538,13.461538,32.692308,2.428571,0.411765,4.25,1.75
2.290554e+14,1,3,2,7,18,5.555556,16.666667,11.111111,38.888889,2.333333,0.285714,7.00,3.00
2.656471e+14,2,7,5,13,43,4.651163,16.279070,11.627907,30.232558,1.857143,0.384615,6.50,3.50
2.813105e+14,5,8,10,15,55,9.090909,14.545455,18.181818,27.272727,1.875000,0.666667,3.00,1.60
2.996404e+14,1,7,11,22,67,1.492537,10.447761,16.417910,32.835821,3.142857,0.500000,22.00,7.00


In [38]:
# save the result to the orginal csv. file
merged_df = pd.merge(health_social, pos_result[['ADJ', 'VERB', 'PRON', 'NOUN', 'total_pos',
                                                'ADJ%', 'VERB%', 'PRON%','NOUN%',
                                                'Noun_to_Verb', 'Pron_to_Noun', 'Noun_to_Adj', 'Verb_to_Adj']],
                     on='id',
                     how='left')
merged_df

,idx,statistics.like_count,link_attachment.caption,statistics.love_count,post_owner.username,post_owner.name,modified_time,statistics.haha_count,surface.id,lang,...,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
0,0,67.0,aarp.org,0.0,AARP,AARP,2024-10-02T10:48:06-07:00,0.0,6.971043e+14,en,...,14.0,47.0,6.382979,19.148936,4.255319,29.787234,1.555556,0.142857,4.666667,3.000000
1,1,96.0,aarp.org,3.0,AARP,AARP,2024-10-02T10:53:20-07:00,2.0,6.971043e+14,en,...,27.0,101.0,5.940594,16.831683,16.831683,26.732673,1.588235,0.629630,4.500000,2.833333
2,2,488.0,NaN,51.0,AARP,AARP,2024-10-02T10:05:42-07:00,2.0,6.971043e+14,en,...,18.0,59.0,5.084746,18.644068,18.644068,30.508475,1.636364,0.611111,6.000000,3.666667
3,3,113.0,NaN,19.0,AARP,AARP,2024-10-01T06:34:55-07:00,0.0,6.971043e+14,en,...,9.0,31.0,3.225806,12.903226,22.580645,29.032258,2.250000,0.777778,9.000000,4.000000
4,4,318.0,NaN,74.0,AARP,AARP,2024-09-24T13:01:49-07:00,1.0,6.971043e+14,en,...,16.0,71.0,9.859155,16.901408,22.535211,22.535211,1.333333,1.000000,2.285714,1.714286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3068,3068,1.0,generations.asaging.org,0.0,asaging,American Society on Aging,2024-05-29T19:03:18-07:00,0.0,1.180081e+15,en,...,9.0,42.0,11.904762,16.666667,2.380952,21.428571,1.285714,0.111111,1.800000,1.400000
3069,3069,0.0,generations.asaging.org,0.0,asaging,American Society on Aging,2024-05-28T05:46:32-07:00,0.0,1.180081e+15,en,...,10.0,44.0,13.636364,13.636364,11.363636,22.727273,1.666667,0.500000,1.666667,1.000000
3070,3070,0.0,generations.asaging.org,0.0,asaging,American Society on Aging,2024-05-28T05:45:39-07:00,0.0,1.180081e+15,en,...,17.0,48.0,4.166667,10.416667,10.416667,35.416667,3.400000,0.294118,8.500000,2.500000
3071,3071,2.0,NaN,0.0,asaging,American Society on Aging,2024-09-07T06:58:41-07:00,0.0,1.180081e+15,en,...,5.0,10.0,10.000000,10.000000,0.000000,50.000000,5.000000,0.000000,5.000000,1.000000


In [40]:
# save the results to a new cvs. file
merged_df.to_csv(brief + 'Health_social_result.csv', index=False)

# AD (clinical label group)


In [41]:
#read the data
ad_clinical = pd.read_csv(result + 'AD_clinical_result.csv',engine='python')

In [42]:
ad_clinical.head(2)

,idx,PAR,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,...,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause,average_similarity
0,0,dementia-001-2,He_Hinzen,ModerateAD,59,male,mhm . there's a young boy going in a cookie ja...,0.029,0.951,0.020,...,110.0,59.0,0.536364,11.0,0.916667,9.0,0.75,3.0,0.25,0.256047
1,1,dementia-003-0,He_Hinzen,MildAD,56,male,here's a cookie jar . and the lid is off the c...,0.013,0.916,0.071,...,189.0,87.0,0.460317,16.0,0.800000,20.0,1.00,3.0,0.15,0.249428


In [43]:
ad_clinical.rename(columns={'PAR': 'id'}, inplace=True)

In [44]:
len(ad_clinical["id"]) == len(set(ad_clinical["id"])) #each id is a unique number

True

In [45]:
# Tokenize the text (split each sentence into words)
ad_clinical['word_list'] = ad_clinical['clean_text'].astype(str).apply(lambda x: x.split())
ad_clinical.head(2)

,idx,id,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,...,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause,average_similarity,word_list
0,0,dementia-001-2,He_Hinzen,ModerateAD,59,male,mhm . there's a young boy going in a cookie ja...,0.029,0.951,0.020,...,59.0,0.536364,11.0,0.916667,9.0,0.75,3.0,0.25,0.256047,"[mhm, ., there's, a, young, boy, going, in, a,..."
1,1,dementia-003-0,He_Hinzen,MildAD,56,male,here's a cookie jar . and the lid is off the c...,0.013,0.916,0.071,...,87.0,0.460317,16.0,0.800000,20.0,1.00,3.0,0.15,0.249428,"[here's, a, cookie, jar, ., and, the, lid, is,..."


In [46]:
df=ad_clinical[['idx',"id",'clean_text','word_list']]
df

,idx,id,clean_text,word_list
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,"[mhm, ., there's, a, young, boy, going, in, a,..."
1,1,dementia-003-0,here's a cookie jar . and the lid is off the c...,"[here's, a, cookie, jar, ., and, the, lid, is,..."
2,2,dementia-005-2,okay he's falling off a chair . she's running ...,"[okay, he's, falling, off, a, chair, ., she's,..."
3,3,dementia-007-3,well the girl is telling the boy to get the co...,"[well, the, girl, is, telling, the, boy, to, g..."
4,4,dementia-010-0,oh boy . wowie the boy's going up on a cookiej...,"[oh, boy, ., wowie, the, boy's, going, up, on,..."
...,...,...,...,...
190,190,Baycrest11976,alright. you're going. uh the reason I'm laugh...,"[alright., you're, going., uh, the, reason, I'..."
191,191,Baycrest12257,right. oh gosh. I don't really know. I guess m...,"[right., oh, gosh., I, don't, really, know., I..."
192,192,Baycrest12813,thinking back tell you a story. I have an inte...,"[thinking, back, tell, you, a, story., I, have..."
193,193,Baycrest7352,can you repeat the question? okay well as I si...,"[can, you, repeat, the, question?, okay, well,..."


In [47]:
# Explode the 'word_list' column, creating a separate row for each word, and renames the exploded column to 'word'
df_exploded = df.explode('word_list').rename(columns={'word_list': 'word'})
df_exploded.head()

,idx,id,clean_text,word
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,mhm
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,.
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,there's
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,a
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,young


In [48]:
# Remove rows where the 'word' column is not alphabetic
df_exploded = df_exploded[df_exploded['word'].str.isalpha()]
df_exploded.head()

,idx,id,clean_text,word
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,mhm
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,a
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,young
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,boy
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,going


In [49]:
# use Spacy: POS tagging
df_lex = process_word_spacy(df_exploded)
df_lex.head()

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,mhm,[mhm],1,[mhm],[INTJ],[UH]
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,a,[a],1,[a],[PRON],[DT]
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,young,[young],1,[young],[ADJ],[JJ]
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,boy,[boy],1,[boy],[PROPN],[NNP]
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,going,[going],1,[go],[VERB],[VBG]


In [50]:
# save the results
df_lex.to_csv(result + 'AD_clinical_lexicon.csv', index=False)

In [51]:
# Count the number of POS tags (group by id)
df_pos = count_pos_tags_by_speaker(df_lex)
df_pos.head()

sp_pos,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,VERB,X,total_pos
id,,,,,,,,,,,,,,,,,
01-1,9,44,51,25,50,0,11,64,7,10,129,9,0,6,76,1,492
05-1,15,18,13,5,16,0,17,36,6,8,64,8,0,2,49,0,257
06-1,17,68,48,47,61,0,76,92,10,33,202,14,0,13,131,0,812
09-1,18,34,45,24,56,0,32,59,7,24,142,8,0,8,86,0,543
10-1,30,73,71,40,95,0,39,94,6,24,279,23,0,2,140,0,916


In [52]:
# count proportion of Part-of-speech
pos_result = pos_analysis(df_pos)
pos_result.head()

sp_pos,ADJ,VERB,PRON,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
id,,,,,,,,,,,,,
01-1,9,76,129,64,492,1.829268,15.447154,26.219512,13.008130,0.842105,2.015625,7.111111,8.444444
05-1,15,49,64,36,257,5.836576,19.066148,24.902724,14.007782,0.734694,1.777778,2.400000,3.266667
06-1,17,131,202,92,812,2.093596,16.133005,24.876847,11.330049,0.702290,2.195652,5.411765,7.705882
09-1,18,86,142,59,543,3.314917,15.837937,26.151013,10.865562,0.686047,2.406780,3.277778,4.777778
10-1,30,140,279,94,916,3.275109,15.283843,30.458515,10.262009,0.671429,2.968085,3.133333,4.666667


In [53]:
# save the result to the orginal csv. file
merged_df = pd.merge(ad_clinical, pos_result[['ADJ', 'VERB', 'PRON', 'NOUN', 'total_pos',
                                              'ADJ%', 'VERB%', 'PRON%','NOUN%',
                                              'Noun_to_Verb', 'Pron_to_Noun', 'Noun_to_Adj', 'Verb_to_Adj']],
                     on='id',
                     how='left')
merged_df

,idx,id,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,...,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
0,0,dementia-001-2,He_Hinzen,ModerateAD,59,male,mhm . there's a young boy going in a cookie ja...,0.029,0.951,0.020,...,10,99,2.020202,18.181818,25.252525,10.101010,0.555556,2.500000,5.000000,9.000000
1,1,dementia-003-0,He_Hinzen,MildAD,56,male,here's a cookie jar . and the lid is off the c...,0.013,0.916,0.071,...,23,179,1.675978,10.614525,30.167598,12.849162,1.210526,2.347826,7.666667,6.333333
2,2,dementia-005-2,He_Hinzen,MildAD,55,male,okay he's falling off a chair . she's running ...,0.222,0.723,0.055,...,4,19,0.000000,15.789474,21.052632,21.052632,1.333333,1.000000,inf,inf
3,3,dementia-007-3,He_Hinzen,ModerateAD,75,female,well the girl is telling the boy to get the co...,0.054,0.893,0.053,...,13,81,0.000000,19.753086,23.456790,16.049383,0.812500,1.461538,inf,inf
4,4,dementia-010-0,He_Hinzen,MildAD,66,male,oh boy . wowie the boy's going up on a cookiej...,0.006,0.949,0.044,...,34,208,2.403846,20.192308,25.000000,16.346154,0.809524,1.529412,6.800000,8.400000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,190,Baycrest11976,Baycrest,MCI,88;00.,male,alright. you're going. uh the reason I'm laugh...,0.042,0.869,0.089,...,96,890,2.696629,16.516854,30.674157,10.786517,0.653061,2.843750,4.000000,6.125000
191,191,Baycrest12257,Baycrest,MCI,74;00.,female,right. oh gosh. I don't really know. I guess m...,0.024,0.898,0.078,...,143,1345,1.933086,15.762082,23.643123,10.631970,0.674528,2.223776,5.500000,8.153846
192,192,Baycrest12813,Baycrest,MCI,66;00.,male,thinking back tell you a story. I have an inte...,0.050,0.844,0.105,...,194,1700,2.470588,17.941176,30.235294,11.411765,0.636066,2.649485,4.619048,7.261905
193,193,Baycrest7352,Baycrest,MCI,71;00.,female,can you repeat the question? okay well as I si...,0.034,0.863,0.103,...,58,356,3.370787,16.011236,25.000000,16.292135,1.017544,1.534483,4.833333,4.750000


In [54]:
# save the results to a new cvs. file
merged_df.to_csv(brief + 'AD_clinical_result.csv', index=False)

# Health (clinical label group)

In [55]:
#read the data
health_clinical = pd.read_csv(result + 'Health_clinical_result.csv',engine='python')

In [56]:
health_clinical.columns

Index(['idx', 'PAR', 'corpus', 'diag', 'age', 'gender', 'clean_text',
       'VADER_neg', 'VADER_neu', 'VADER_pos', 'VADER_compound',
       'VADER_sentiment', 'sent_count', 'word_count', 'unique_word_count',
       'TTR', 'total_dependents', 'dependents_per_clause',
       'total_coord_phrases', 'coord_phrases_per_clause',
       'total_complex_nominals', 'complex_nominals_per_clause',
       'average_similarity'],
      dtype='object')

In [57]:
health_clinical.head(2)

,idx,PAR,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,...,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause,average_similarity
0,0,control-002-0,He_Hinzen,Control,58,female,the scene is in the kitchen . the mother is wi...,0.01,0.925,0.064,...,133.0,80.0,0.601504,8.0,0.444444,3.0,0.166667,5.0,0.277778,0.228958
1,1,control-013-0,He_Hinzen,Control,62,female,somebody's getting cookies out of the cookie j...,0.02,0.980,0.000,...,88.0,56.0,0.636364,4.0,0.333333,6.0,0.500000,2.0,0.166667,0.255113


In [58]:
health_clinical.rename(columns={'PAR': 'id'}, inplace=True)

In [59]:
len(health_clinical["id"]) == len(set(health_clinical["id"])) #each id is a unique number

True

In [60]:
# Tokenize the text (split each sentence into words)
health_clinical['word_list'] = health_clinical['clean_text'].astype(str).apply(lambda x: x.split())
health_clinical.head(2)

,idx,id,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,...,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause,average_similarity,word_list
0,0,control-002-0,He_Hinzen,Control,58,female,the scene is in the kitchen . the mother is wi...,0.01,0.925,0.064,...,80.0,0.601504,8.0,0.444444,3.0,0.166667,5.0,0.277778,0.228958,"[the, scene, is, in, the, kitchen, ., the, mot..."
1,1,control-013-0,He_Hinzen,Control,62,female,somebody's getting cookies out of the cookie j...,0.02,0.980,0.000,...,56.0,0.636364,4.0,0.333333,6.0,0.500000,2.0,0.166667,0.255113,"[somebody's, getting, cookies, out, of, the, c..."


In [61]:
df=health_clinical[['idx','id','clean_text','word_list']]
df

,idx,id,clean_text,word_list
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,"[the, scene, is, in, the, kitchen, ., the, mot..."
1,1,control-013-0,somebody's getting cookies out of the cookie j...,"[somebody's, getting, cookies, out, of, the, c..."
2,2,control-015-1,okay . a little boy is stepping on a stool tha...,"[okay, ., a, little, boy, is, stepping, on, a,..."
3,3,control-017-4,are you ready ? well the sink is overflowing ....,"[are, you, ready, ?, well, the, sink, is, over..."
4,4,control-021-1,okay . the mother's washing the dishes and the...,"[okay, ., the, mother's, washing, the, dishes,..."
...,...,...,...,...
99,99,94-1,yep,[yep]
100,100,96-1,yep,[yep]
101,101,97-1,yeah,[yeah]
102,102,98-1,okay,[okay]


In [62]:
# Explode the 'word_list' column, creating a separate row for each word, and renames the exploded column to 'word'
df_exploded = df.explode('word_list').rename(columns={'word_list': 'word'})
df_exploded.head()

,idx,id,clean_text,word
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,the
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,scene
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,is
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,in
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,the


In [63]:
# Remove rows where the 'word' column is not alphabetic
df_exploded = df_exploded[df_exploded['word'].str.isalpha()]
df_exploded.head()

,idx,id,clean_text,word
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,the
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,scene
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,is
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,in
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,the


In [64]:
# use Spacy: POS tagging
df_lex = process_word_spacy(df_exploded)
df_lex.head()

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,the,[the],1,[the],[PRON],[DT]
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,scene,[scene],1,[scene],[NOUN],[NN]
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,is,[is],1,[be],[AUX],[VBZ]
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,in,[in],1,[in],[ADP],[IN]
0,0,control-002-0,the scene is in the kitchen . the mother is wi...,the,[the],1,[the],[PRON],[DT]


In [65]:
# save the results
df_lex.to_csv(result + 'Health_clinical_lexicon.csv', index=False)

In [66]:
# Count the number of POS tags (group by id)
df_pos = count_pos_tags_by_speaker(df_lex)
df_pos.head()

sp_pos,ADJ,ADP,ADV,AUX,CCONJ,INTJ,NOUN,NUM,PART,PRON,PROPN,SCONJ,VERB,total_pos
id,,,,,,,,,,,,,,
02-1,2,5,1,0,0,3,4,0,1,4,2,0,4,26
03-1,0,1,1,1,0,0,4,0,0,1,0,0,2,10
04-1,0,0,0,0,0,1,0,0,0,0,0,0,0,1
07-1,1,1,0,0,0,1,2,0,0,3,1,0,2,11
08-1,0,0,1,0,0,1,1,0,1,1,0,0,0,5


In [67]:
# count proportion of Part-of-speech
pos_result = pos_analysis(df_pos)
pos_result.head()

sp_pos,ADJ,VERB,PRON,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
id,,,,,,,,,,,,,
02-1,2,4,4,4,26,7.692308,15.384615,15.384615,15.384615,1.0,1.00,2.0,2.0
03-1,0,2,1,4,10,0.000000,20.000000,10.000000,40.000000,2.0,0.25,inf,inf
04-1,0,0,0,0,1,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
07-1,1,2,3,2,11,9.090909,18.181818,27.272727,18.181818,1.0,1.50,2.0,2.0
08-1,0,0,1,1,5,0.000000,0.000000,20.000000,20.000000,inf,1.00,inf,NaN


In [68]:
pos_result.columns

Index(['ADJ', 'VERB', 'PRON', 'NOUN', 'total_pos', 'ADJ%', 'VERB%', 'PRON%',
       'NOUN%', 'Noun_to_Verb', 'Pron_to_Noun', 'Noun_to_Adj', 'Verb_to_Adj'],
      dtype='object', name='sp_pos')

In [69]:
# save the result to the orginal csv. file
merged_df = pd.merge(health_clinical, pos_result[['ADJ', 'VERB', 'PRON', 'NOUN', 'total_pos',
                                                  'ADJ%', 'VERB%', 'PRON%','NOUN%',
                                                  'Noun_to_Verb', 'Pron_to_Noun', 'Noun_to_Adj', 'Verb_to_Adj']],
                     on='id',
                     how='left')
merged_df

,idx,id,corpus,diag,age,gender,clean_text,VADER_neg,VADER_neu,VADER_pos,...,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
0,0,control-002-0,He_Hinzen,Control,58,female,the scene is in the kitchen . the mother is wi...,0.010,0.925,0.064,...,26,124,6.451613,16.935484,20.967742,20.967742,1.238095,1.000000,3.250,2.625
1,1,control-013-0,He_Hinzen,Control,62,female,somebody's getting cookies out of the cookie j...,0.020,0.980,0.000,...,16,78,0.000000,21.794872,23.076923,20.512821,0.941176,1.125000,inf,inf
2,2,control-015-1,He_Hinzen,Control,67,female,okay . a little boy is stepping on a stool tha...,0.029,0.923,0.048,...,33,152,3.289474,13.157895,22.368421,21.710526,1.650000,1.030303,6.600,4.000
3,3,control-017-4,He_Hinzen,Control,71,female,are you ready ? well the sink is overflowing ....,0.090,0.827,0.083,...,33,221,3.619910,19.004525,24.434389,14.932127,0.785714,1.636364,4.125,5.250
4,4,control-021-1,He_Hinzen,Control,74,female,okay . the mother's washing the dishes and the...,0.007,0.963,0.030,...,24,158,2.531646,18.354430,27.215190,15.189873,0.827586,1.791667,6.000,7.250
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,99,94-1,Delaware,Control,68;00.,female,yep,0.000,0.000,1.000,...,0,1,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
100,100,96-1,Delaware,Control,61;00.,female,yep,0.000,0.000,1.000,...,0,1,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
101,101,97-1,Delaware,Control,84;00.,female,yeah,0.000,0.000,1.000,...,0,1,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
102,102,98-1,Delaware,Control,63;00.,female,okay,0.000,0.000,1.000,...,0,1,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN


In [70]:
# save the results to a new cvs. file
merged_df.to_csv(brief + 'Health_clinical_result.csv', index=False)